# HW4 — Monthly retraining: timing and strategy comparison
**Author:** Mauricio Gutierrez

The existing model refits once per year, not only once in 2013. This notebook
compares that schedule with a fresh fit each month. Use the same seven features
from notebook 04. Keep the trading rules and fees fixed.

```text
Past signals with completed 90-day outcomes -> Expanding training window
                                                       |
                                                       v
                                          Refit before each month's signal
                                                       |
                                                       v
                                   New prices -> Features -> Updated forecasts
                                                       |
                                                       v
                                           Same monthly trading strategy
```

Monthly fitting uses newly completed outcomes. It cannot use last month's
90-day returns before they finish. It uses all eligible stock/date rows, not
only executed purchases. Each fit reuses older eligible rows and adds newly
available rows. It refits the entire preprocessing and Ridge pipeline.

## Run instructions
1. Run notebooks 01 through 04 if their saved outputs are missing or outdated.
2. Open `05_monthly_retraining.ipynb` with the repository `.venv` kernel.
3. Select **Restart Kernel and Run All Cells**.
4. Read the timing table and training-cutoff log.
5. Read the strategy comparison.

The run order is **01 -> 02 -> 03 -> 04 -> 05**. This notebook saves a separate
experiment. It does not replace the earlier annual-refit results. Restart the
kernel after editing `feature_engineering.py`.

> **Development step, class data (30 stocks).** Monthly against annual refits; annual was kept. The final strategy is defined in `run_final_test.py` and reported in `07_final_test_report.ipynb`.

In [ ]:
from pathlib import Path
from time import perf_counter
from dataclasses import asdict
import hashlib
import json
import sys
import io
import unittest
import platform

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "Lectures/modeling/course_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the kernel inside the course repository.")
HW_DIR = ROOT / "Homeworks/HW4/Mauricio_Gutierrez"
if str(HW_DIR) not in sys.path:
    sys.path.insert(0, str(HW_DIR))
from feature_engineering import (
    load_cleaned_data, price_features, additional_price_features, apply_membership,
    make_labels, annual_split, make_ridge_pipeline, walk_forward_forecasts,
)
from strategy_backtest import BacktestConfig, Backtester, compare_performance

DATABASE = HW_DIR / "outputs/data_cleaning/equities_class/equities_class_cleaned.duckdb"
PREVIOUS_DIR = HW_DIR / "outputs/feature_experiments/monthly_volatility_rsi"
OUTPUT_DIR = HW_DIR / "outputs/retraining/monthly_vs_annual"
REPEATS = 3
SAVE_OUTPUTS = True

def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

if not (PREVIOUS_DIR / "experiment.json").is_file():
    raise FileNotFoundError("Run notebook 04 before this notebook.")
settings = json.loads((PREVIOUS_DIR / "experiment.json").read_text(encoding="utf-8"))
assert digest(DATABASE) == settings["database_sha256"]
if digest(HW_DIR / "strategy_backtest.py") != settings["backtester_sha256"]:
    print("The shared engine changed. The saved-baseline reproduction checks below must still pass.")
protected = {p: digest(p) for p in PREVIOUS_DIR.rglob("*") if p.is_file()}
protected[DATABASE] = digest(DATABASE)
FEATURES = settings["expanded_features"]
CONFIG = BacktestConfig(**settings["config"])
MODEL = settings["model_settings"]
START, END = settings["start"], settings["end"]
assert pd.Timestamp(END) <= pd.Timestamp("2015-12-31")

## 1. Prepare shared inputs once
Both schedules use the same cleaned 30-stock dataset, seven features, target
definition, and 2013–2015 period. Time data loading and panel preparation
separately. These operations are shared; monthly refitting does not require
recleaning the database 36 times.

Features use data through the signal close. Labels can contain future outcomes
for retrospective evaluation. The split always checks that the label ended
strictly before the fit cutoff before allowing that row into training.

In [2]:
before_load = perf_counter()
prices, calendar, membership = load_cleaned_data(DATABASE, end=MODEL["data_end"])
with duckdb.connect(str(DATABASE), read_only=True) as connection:
    universe = connection.execute("SELECT symbol, sector FROM universe").fetchdf()
    benchmark = connection.execute("SELECT date, adj_close FROM benchmark WHERE symbol='^SP500TR' AND date <= ? ORDER BY date", [END]).fetchdf()
sectors = universe.set_index("symbol").sector.to_dict()
benchmark["date"] = pd.to_datetime(benchmark.date).astype("datetime64[ns]")
load_seconds = perf_counter() - before_load

before_panel = perf_counter()
base = apply_membership(price_features(prices, calendar), membership, calendar)
panel = base.merge(additional_price_features(prices, calendar), on=["date", "symbol"], validate="one_to_one")
panel = make_labels(panel, prices, calendar, holding_days=CONFIG.holding_days,
                    max_exit_delay=MODEL["max_exit_delay_sessions"])
panel_seconds = perf_counter() - before_panel
shared_timing = pd.Series({"load_seconds": load_seconds, "features_and_labels_seconds": panel_seconds}, name="Shared preparation")
display(shared_timing.to_frame())
print(f"Stocks: {prices.symbol.nunique()}; features: {len(FEATURES)}; evaluation: {START} to {END}")

Stocks: 30; features: 7; evaluation: 2013-01-01 to 2015-12-31


,Shared preparation
load_seconds,0.080719
features_and_labels_seconds,0.522272


## 2. Measure annual and monthly refitting
Run one small warm-up fit before measuring. Run each complete schedule three
times. Alternate the order to reduce first-run effects. Report the median,
minimum, and maximum elapsed times. Hardware load and caching affect results.

The measured operation includes splitting, copying model inputs, fitting
clipping/median/scaling/Ridge, predicting, and assembling forecast output.
Both schedules predict all the same daily rows. Annual fitting does this with
three fits. Monthly fitting uses 36 fits. No hyperparameter search is included.

Imports, data preparation, tests, backtesting, display, and file export are
outside this measured operation. The fit log also times pipeline `fit` and
`predict` separately. A normal experiment needs one schedule run, not three
benchmark repetitions.

In [3]:
warmup_train, _ = annual_split(panel, 2013, train_start=MODEL["train_start"])
warmup = make_ridge_pipeline(alpha=MODEL["alpha"], feature_names=FEATURES)
warmup.fit(warmup_train[FEATURES], warmup_train.target)

measurements, forecasts_by_schedule, logs = [], {}, {}
for repeat in range(REPEATS):
    order = ["annual", "monthly"] if repeat % 2 == 0 else ["monthly", "annual"]
    for frequency in order:
        before = perf_counter()
        forecasts, fit_log = walk_forward_forecasts(
            panel, FEATURES, start=START, end=END, train_start=MODEL["train_start"],
            frequency=frequency, alpha=MODEL["alpha"],
        )
        elapsed = perf_counter() - before
        measurements.append({"repeat": repeat + 1, "schedule": frequency,
                             "elapsed_seconds": elapsed, "fits": len(fit_log),
                             "forecast_rows": len(forecasts),
                             "pipeline_fit_seconds": fit_log.fit_seconds.sum(),
                             "pipeline_predict_seconds": fit_log.predict_seconds.sum()})
        if frequency in forecasts_by_schedule:
            pd.testing.assert_frame_equal(forecasts, forecasts_by_schedule[frequency])
        forecasts_by_schedule[frequency] = forecasts
        logs[frequency] = fit_log

timings = pd.DataFrame(measurements)
timing_summary = timings.groupby("schedule").agg(
    fits=("fits", "first"), forecast_rows=("forecast_rows", "first"),
    median_seconds=("elapsed_seconds", "median"),
    min_seconds=("elapsed_seconds", "min"), max_seconds=("elapsed_seconds", "max"),
    median_fit_seconds=("pipeline_fit_seconds", "median"),
    median_predict_seconds=("pipeline_predict_seconds", "median"),
)
display(timing_summary.round(4))
assert len(logs["annual"]) == 3 and len(logs["monthly"]) == 36
pd.testing.assert_frame_equal(forecasts_by_schedule["annual"][["date", "symbol"]],
                              forecasts_by_schedule["monthly"][["date", "symbol"]])
saved = pd.read_csv(PREVIOUS_DIR / "expanded/forecasts.csv", parse_dates=["date"])
check = forecasts_by_schedule["annual"].merge(saved[["date", "symbol", "prediction"]],
    on=["date", "symbol"], suffixes=("_new", "_saved"), how="outer", indicator=True, validate="one_to_one")
assert check._merge.eq("both").all()
np.testing.assert_allclose(check.prediction_new, check.prediction_saved, atol=1e-10, rtol=1e-10)
print("PASS: annual forecasts reproduce notebook 04; both schedules forecast identical rows.")

PASS: annual forecasts reproduce notebook 04; both schedules forecast identical rows.


,fits,forecast_rows,median_seconds,min_seconds,max_seconds,median_fit_seconds,median_predict_seconds
schedule,,,,,,,
annual,3,21578,0.1138,0.1046,0.1217,0.0804,0.0088
monthly,36,21578,1.2573,1.2132,1.3280,1.0280,0.0768


## 3. Inspect the monthly training window
Read `last_training_outcome` beside `fit_cutoff`. The outcome must always end
earlier. `new_training_rows` counts all rows newly available since the previous
fit. For the first fit, it counts the entire initial training set.

```text
January fit -> Old outcomes completed before January signal
February fit -> Same history + Outcomes completed before February signal
March fit -> Same history + Further completed outcomes
```

The model refits on the first eligible signal session of each month. In this
dataset that matches the strategy's first market session. Purchases still occur
at the next close. Refit even if that month has no cash to buy stocks.

In [4]:
monthly_log = logs["monthly"]
assert monthly_log.last_training_outcome.lt(monthly_log.fit_cutoff).all()
assert monthly_log.training_rows.diff().dropna().ge(0).all()
first_sessions = calendar.to_series().groupby(calendar.to_period("M")).first()
expected = first_sessions.loc[(first_sessions >= START) & (first_sessions <= END)]
assert monthly_log.fit_cutoff.tolist() == expected.tolist()
display(monthly_log[["period", "fit_cutoff", "last_training_signal", "last_training_outcome",
                     "training_rows", "new_training_rows", "forecast_rows"]].head(6))
display(monthly_log.tail(3))

,period,fit_cutoff,last_training_signal,last_training_outcome,training_rows,new_training_rows,forecast_rows
0,2013-01,2013-01-02,2012-10-01,2012-12-31,20034,20034,609
1,2013-02,2013-02-01,2012-11-01,2013-01-31,20643,609,551
2,2013-03,2013-03-01,2012-11-29,2013-02-28,21194,551,580
3,2013-04,2013-04-01,2012-12-27,2013-03-28,21745,551,638
4,2013-05,2013-05-01,2013-01-29,2013-04-30,22354,609,638
5,2013-06,2013-06-03,2013-02-28,2013-05-30,22963,609,580


,period,fit_cutoff,last_training_signal,last_training_outcome,training_rows,new_training_rows,forecast_rows,fit_seconds,predict_seconds
33,2015-10,2015-10-01,2015-07-01,2015-09-30,39917,636,594,0.033732,0.002011
34,2015-11,2015-11-02,2015-07-30,2015-10-29,40477,560,540,0.036326,0.001989
35,2015-12,2015-12-01,2015-08-31,2015-11-30,41093,616,567,0.034323,0.002153


## 4. Check the timing rules and compare portfolios
Run the synthetic tests, including a case where a target becomes available
between January and February. It must affect February's fit but not January's.
Then run the unchanged backtester once for each forecast schedule. The portfolio
time below is measured separately from fitting. Refitting more often does not
guarantee a better return.

In [5]:
suite = unittest.defaultTestLoader.discover(str(HW_DIR), pattern="test_*.py")
test_log = io.StringIO()
test_result = unittest.TextTestRunner(stream=test_log, verbosity=1).run(suite)
print(test_log.getvalue())
assert test_result.wasSuccessful()

accounts, performance_parts, backtest_timings = {}, [], []
for frequency, forecasts in forecasts_by_schedule.items():
    before = perf_counter()
    account = Backtester(CONFIG).run(prices, calendar, forecasts, sectors, membership, START, END)
    curves, scores = compare_performance(account.daily, benchmark, CONFIG)
    elapsed = perf_counter() - before
    accounts[frequency] = account
    performance_parts.append(scores.loc[["Strategy"]].rename(index={"Strategy": frequency}))
    backtest_timings.append({"schedule": frequency, "backtest_seconds": elapsed})
    flows = account.trades.groupby("date").cash_change.sum().reindex(account.daily.date, fill_value=0.0)
    np.testing.assert_allclose(account.daily.cash, CONFIG.initial_cash + flows.cumsum(), atol=1e-6)
    assert np.isclose(account.daily.nav.iloc[-1], account.daily.cash.iloc[-1] + account.open_lots.market_value.sum())
performance_parts.append(scores.loc[["S&P 500 TR"]])
performance = pd.concat(performance_parts)
backtest_timing = pd.DataFrame(backtest_timings).set_index("schedule")
saved_perf = pd.read_csv(PREVIOUS_DIR / "performance_comparison.csv", index_col=0)
np.testing.assert_allclose(performance.loc["annual"], saved_perf.loc["Ridge + volatility + RSI"], rtol=1e-10, atol=1e-8)
display(backtest_timing.round(4))
formatted = performance.copy()
for column, pattern in {"final_value": "${:,.2f}", "total_return": "{:.2%}", "CAGR": "{:.2%}",
                         "max_drawdown": "{:.2%}", "daily_Sharpe": "{:.3f}"}.items():
    formatted[column] = performance[column].map(pattern.format)
display(formatted)

..........................
----------------------------------------------------------------------
Ran 26 tests in 0.625s

OK



,backtest_seconds
schedule,
annual,0.2886
monthly,0.2903


,final_value,total_return,CAGR,max_drawdown,daily_Sharpe
portfolio,,,,,
annual,"$1,322,879.91",32.29%,9.80%,-29.94%,0.391
monthly,"$1,245,508.11",24.55%,7.61%,-29.13%,0.328
S&P 500 TR,"$1,487,843.34",48.78%,14.20%,-12.04%,0.950


## 5. Read the measured cost
The conclusion uses median measured times. Shared preparation is timed once.
The estimate for core stages adds preparation, one forecast-schedule run, and
one backtest. It excludes Python startup, notebook rendering, tests, benchmark
repetitions, and export. Do not use it as a promise for full-notebook wall time.
These measurements apply to the 30-stock class dataset on this machine. They
do not estimate the full 3,849-stock database or a parameter-search workload.

In [6]:
annual_seconds = timing_summary.loc["annual", "median_seconds"]
monthly_seconds = timing_summary.loc["monthly", "median_seconds"]
extra_seconds = monthly_seconds - annual_seconds
ratio = monthly_seconds / annual_seconds
core_estimate = timing_summary[["median_seconds"]].join(backtest_timing)
core_estimate["shared_preparation_seconds"] = load_seconds + panel_seconds
core_estimate["core_seconds_estimate"] = core_estimate.sum(axis=1)
display(core_estimate.round(4))
analysis_text = f"""### Monthly retraining cost on the class dataset

The existing schedule fits **3 times**. Monthly retraining fits **36 times**.
Both schedules forecast **{int(timing_summary.loc['monthly', 'forecast_rows']):,} rows**.
The median fit-and-forecast loop took **{annual_seconds:.3f} seconds** annually and
**{monthly_seconds:.3f} seconds** monthly. Monthly retraining added
**{extra_seconds:.3f} seconds**, or **{ratio:.2f} times** the annual loop's time.

Shared data preparation took **{load_seconds + panel_seconds:.3f} seconds**.
The estimated monthly core run, including preparation and one backtest, took
**{core_estimate.loc['monthly', 'core_seconds_estimate']:.3f} seconds**. This estimate
excludes imports, tests, repeated timing trials, display, and file export.

Monthly retraining is practical for this small three-year experiment. It uses
newly completed 90-day outcomes, not unfinished outcomes from the latest month.

For context, annual-refit total return was **{performance.loc['annual', 'total_return']:.2%}**
and monthly-refit return was **{performance.loc['monthly', 'total_return']:.2%}**.
Annualized daily-return Sharpe was **{performance.loc['annual', 'daily_Sharpe']:.3f}**
and **{performance.loc['monthly', 'daily_Sharpe']:.3f}**, respectively.
These are development results. The final test remains closed.
"""
display(Markdown(analysis_text))

,median_seconds,backtest_seconds,shared_preparation_seconds,core_seconds_estimate
schedule,,,,
annual,0.1138,0.2886,0.603,1.0054
monthly,1.2573,0.2903,0.603,2.1506


### Monthly retraining cost on the class dataset

The existing schedule fits **3 times**. Monthly retraining fits **36 times**.
Both schedules forecast **21,578 rows**.
The median fit-and-forecast loop took **0.114 seconds** annually and
**1.257 seconds** monthly. Monthly retraining added
**1.144 seconds**, or **11.05 times** the annual loop's time.

Shared data preparation took **0.603 seconds**.
The estimated monthly core run, including preparation and one backtest, took
**2.151 seconds**. This estimate
excludes imports, tests, repeated timing trials, display, and file export.

Monthly retraining is practical for this small three-year experiment. It uses
newly completed 90-day outcomes, not unfinished outcomes from the latest month.

For context, annual-refit total return was **32.29%**
and monthly-refit return was **24.55%**.
Annualized daily-return Sharpe was **0.391**
and **0.328**, respectively.
These are development results. The final test remains closed.


## 6. Save the separate experiment
Save raw trial timings, timing summaries, the monthly fit log, forecast files,
portfolio records, and the conclusion. Existing notebook 04 files stay unchanged.
The run is reproducible; exact timings will vary with machine load and caching.

In [7]:
assert all(digest(path) == value for path, value in protected.items())
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    timings.to_csv(OUTPUT_DIR / "timing_trials.csv", index=False)
    timing_summary.to_csv(OUTPUT_DIR / "timing_summary.csv")
    shared_timing.to_csv(OUTPUT_DIR / "shared_preparation.csv")
    core_estimate.to_csv(OUTPUT_DIR / "core_time_estimates.csv")
    performance.to_csv(OUTPUT_DIR / "performance_comparison.csv")
    (OUTPUT_DIR / "analysis.md").write_text(analysis_text, encoding="utf-8")
    for frequency in ["annual", "monthly"]:
        target = OUTPUT_DIR / frequency
        target.mkdir(exist_ok=True)
        forecasts_by_schedule[frequency].to_csv(target / "forecasts.csv", index=False)
        logs[frequency].to_csv(target / "fit_log.csv", index=False)
        for field in ["daily", "trades", "orders", "decisions", "rejections", "open_lots"]:
            getattr(accounts[frequency], field).to_csv(target / f"{field}.csv", index=False)
    manifest = {"scope": "class-data development", "start": START, "end": END, "repeats": REPEATS,
                "features": FEATURES, "alpha": MODEL["alpha"], "config": asdict(CONFIG),
                "training_start": MODEL["train_start"], "outcome_rule": "label_end strictly before first signal",
                "python": platform.python_version(), "system": platform.system(), "machine": platform.machine(),
                "database_sha256": digest(DATABASE), "feature_module_sha256": digest(HW_DIR / "feature_engineering.py"),
                "backtester_sha256": digest(HW_DIR / "strategy_backtest.py")}
    (OUTPUT_DIR / "experiment.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Saved: {OUTPUT_DIR.relative_to(ROOT)}")
print("PASS: prior experiment files and the cleaned database are unchanged.")

Saved: Homeworks\HW4\Mauricio_Gutierrez\outputs\retraining\monthly_vs_annual
PASS: prior experiment files and the cleaned database are unchanged.
